# 03 — Feature engineering for 16-day forecasting

This notebook creates leakage-safe features for a global machine-learning model. Each historical cutoff reproduces the real forecasting situation: only information available at the cutoff is used to predict the following 16 days.

Outputs:
- `data/interim/features_train.parquet` — historical training windows
- `data/interim/features_validation.parquet` — final 16 days of `train.csv`
- `data/interim/features_test.parquet` — Kaggle test horizon

## 1. Imports and paths

Place this notebook in the project root. It expects the original Kaggle files under `data/raw/`.

In [10]:
from pathlib import Path
from datetime import timedelta
import polars as pl

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data' / 'raw').exists() and (PROJECT_ROOT.parent / 'data' / 'raw').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
INTERIM_DIR = PROJECT_ROOT / 'data' / 'interim'
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

required = ['train.csv', 'test.csv', 'stores.csv', 'oil.csv', 'holidays_events.csv']
missing = [name for name in required if not (RAW_DIR / name).exists()]
assert not missing, f'Missing files in {RAW_DIR}: {missing}'
print(f'Project root: {PROJECT_ROOT}')
print(f'Polars: {pl.__version__}')

Project root: retail-demand-forecasting
Polars: 1.44.2


## 2. Load source tables

In [11]:
train = pl.read_csv(RAW_DIR / 'train.csv', try_parse_dates=True)
test = pl.read_csv(RAW_DIR / 'test.csv', try_parse_dates=True)
stores = pl.read_csv(RAW_DIR / 'stores.csv')
oil = pl.read_csv(RAW_DIR / 'oil.csv', try_parse_dates=True)
holidays = pl.read_csv(RAW_DIR / 'holidays_events.csv', try_parse_dates=True)

assert train.select(pl.struct(['date', 'store_nbr', 'family']).is_duplicated().any()).item() is False
assert test.select(pl.struct(['date', 'store_nbr', 'family']).is_duplicated().any()).item() is False

print('train:', train.shape, train['date'].min(), 'to', train['date'].max())
print('test: ', test.shape, test['date'].min(), 'to', test['date'].max())
print('stores:', stores.shape, '| oil:', oil.shape, '| holidays:', holidays.shape)

train: (3000888, 6) 2013-01-01 to 2017-08-15
test:  (28512, 5) 2017-08-16 to 2017-08-31
stores: (54, 5) | oil: (1218, 2) | holidays: (350, 6)


## 3. Prepare future-known external features

Oil is forward-filled on a complete daily calendar, using past observations only. Holidays are expanded to the stores they actually affect: national holidays apply to all stores, regional holidays match `state`, and local holidays match `city`. Transferred holiday rows are excluded because they did not occur on their original date.

In [19]:
all_dates = pl.DataFrame({
    'date': pl.date_range(
        min(train['date'].min(), oil['date'].min()),
        max(test['date'].max(), oil['date'].max()),
        interval='1d',
        eager=True,
    )
})

oil_daily = (
    all_dates
    .join(oil.select('date', 'dcoilwtico'), on='date', how='left')
    .sort('date')
    .with_columns(pl.col('dcoilwtico').forward_fill())
)

active_holidays = holidays.filter(~pl.col('transferred').fill_null(False))
store_locations = stores.select('store_nbr', 'city', 'state')

national = (
    active_holidays
    .filter(pl.col("locale") == "National")
    .select("date", "type")
    .join(
        store_locations.select("store_nbr"),
        how="cross",
    )
    .select("date", "store_nbr", "type")
)

regional = (
    active_holidays.filter(pl.col('locale') == 'Regional')
    .join(store_locations, left_on='locale_name', right_on='state', how='inner')
    .select('date', 'store_nbr', 'type')
)
local = (
    active_holidays.filter(pl.col('locale') == 'Local')
    .join(store_locations, left_on='locale_name', right_on='city', how='inner')
    .select('date', 'store_nbr', 'type')
)

holiday_store = (
    pl.concat([national, regional, local], how='vertical')
    .group_by('date', 'store_nbr')
    .agg(
        pl.lit(1).alias('is_holiday'),
        pl.col('type').str.contains('Holiday').any().cast(pl.Int8).alias('has_holiday_type'),
        pl.col('type').str.contains('Event').any().cast(pl.Int8).alias('has_event'),
        pl.len().cast(pl.Int16).alias('holiday_count'),
    )
)

print('Store-date holiday rows:', holiday_store.height)
oil_daily.tail(3)

Store-date holiday rows: 9139


date,dcoilwtico
date,f64
2017-08-29,46.46
2017-08-30,45.96
2017-08-31,47.26


## 4. Leakage-safe feature builder

For a cutoff `C`, targets are dates `C+1` through `C+16`. Rolling statistics use only `C-27` through `C`. Direct sales lags are at least 16 days, so every source date is on or before the cutoff for the entire horizon.

In [15]:
KEYS = ['date', 'store_nbr', 'family']
SERIES_KEYS = ['store_nbr', 'family']
SAFE_LAGS = [16, 21, 28, 35]
HORIZON = 16

def build_window(cutoff, future_rows, history, include_target=True):
    cutoff = cutoff if hasattr(cutoff, 'year') else cutoff.date()
    start = cutoff + timedelta(days=1)
    end = cutoff + timedelta(days=HORIZON)

    window = future_rows.filter(pl.col('date').is_between(start, end)).clone()
    expected = stores.height * train['family'].n_unique() * HORIZON
    assert window.height == expected, f'{cutoff}: expected {expected:,} rows, found {window.height:,}'

    hist28 = history.filter(pl.col('date').is_between(cutoff - timedelta(days=27), cutoff))
    hist14 = history.filter(pl.col('date').is_between(cutoff - timedelta(days=13), cutoff))
    hist7 = history.filter(pl.col('date').is_between(cutoff - timedelta(days=6), cutoff))
    prev7 = history.filter(pl.col('date').is_between(cutoff - timedelta(days=13), cutoff - timedelta(days=7)))

    stats28 = hist28.group_by(SERIES_KEYS).agg(
        pl.col('sales').mean().alias('sales_mean_28'),
        pl.col('sales').std().fill_null(0).alias('sales_std_28'),
        pl.col('sales').min().alias('sales_min_28'),
        pl.col('sales').max().alias('sales_max_28'),
        (pl.col('sales') == 0).mean().alias('zero_rate_28'),
    )
    stats14 = hist14.group_by(SERIES_KEYS).agg(pl.col('sales').mean().alias('sales_mean_14'))
    stats7 = hist7.group_by(SERIES_KEYS).agg(pl.col('sales').mean().alias('sales_mean_7'))
    stats_prev7 = prev7.group_by(SERIES_KEYS).agg(pl.col('sales').mean().alias('sales_mean_prev7'))

    window = (
        window
        .join(stats28, on=SERIES_KEYS, how='left')
        .join(stats14, on=SERIES_KEYS, how='left')
        .join(stats7, on=SERIES_KEYS, how='left')
        .join(stats_prev7, on=SERIES_KEYS, how='left')
        .with_columns((pl.col('sales_mean_7') - pl.col('sales_mean_prev7')).alias('sales_trend_7'))
    )

    for lag in SAFE_LAGS:
        lag_values = history.select(
            (pl.col('date') + pl.duration(days=lag)).alias('date'),
            'store_nbr', 'family',
            pl.col('sales').alias(f'sales_lag_{lag}'),
        )
        window = window.join(lag_values, on=KEYS, how='left')

    window = (
        window
        .join(stores, on='store_nbr', how='left')
        .join(oil_daily, on='date', how='left')
        .join(holiday_store, on=['date', 'store_nbr'], how='left')
        .with_columns(
            pl.lit(cutoff).cast(pl.Date).alias('cutoff_date'),
            ((pl.col('date') - pl.lit(cutoff).cast(pl.Date)).dt.total_days()).cast(pl.Int8).alias('day_of_forecast'),
            pl.col('date').dt.weekday().cast(pl.Int8).alias('day_of_week'),
            pl.col('date').dt.day().cast(pl.Int8).alias('day_of_month'),
            pl.col('date').dt.month().cast(pl.Int8).alias('month'),
            pl.col('date').dt.year().cast(pl.Int16).alias('year'),
            (pl.col('date').dt.weekday() >= 6).cast(pl.Int8).alias('is_weekend'),
            pl.col('date').dt.ordinal_day().cast(pl.Int16).alias('day_of_year'),
            pl.col('is_holiday').fill_null(0).cast(pl.Int8),
            pl.col('has_holiday_type').fill_null(0).cast(pl.Int8),
            pl.col('has_event').fill_null(0).cast(pl.Int8),
            pl.col('holiday_count').fill_null(0).cast(pl.Int16),
        )
    )

    if not include_target and 'sales' in window.columns:
        window = window.drop('sales')
    assert window.select(pl.struct(KEYS).is_duplicated().any()).item() is False
    return window.sort(KEYS)

## 5. Define historical cutoffs

The final 16 days are reserved for validation. Twelve earlier cutoffs, spaced 28 days apart, become training examples. This produces 342,144 training rows while keeping execution practical on a laptop. Increase `N_TRAIN_WINDOWS` later if memory and training time permit.

In [21]:
validation_end = train["date"].max()
validation_cutoff = validation_end - timedelta(days=HORIZON)
test_cutoff = train["date"].max()

N_TRAIN_WINDOWS = 12
WINDOW_SPACING_DAYS = 28

EXPECTED_WINDOW_ROWS = (
    stores.height
    * train["family"].n_unique()
    * HORIZON
)


def has_complete_horizon(cutoff):
    start = cutoff + timedelta(days=1)
    end = cutoff + timedelta(days=HORIZON)

    rows = train.filter(
        pl.col("date").is_between(start, end)
    ).height

    return rows == EXPECTED_WINDOW_ROWS


train_cutoffs = []

for i in range(N_TRAIN_WINDOWS, 0, -1):
    cutoff = validation_cutoff - timedelta(
        days=WINDOW_SPACING_DAYS * i
    )

    original_cutoff = cutoff

    # Avanza el corte hasta encontrar 16 días completos.
    while not has_complete_horizon(cutoff):
        cutoff += timedelta(days=1)

    if cutoff != original_cutoff:
        print(
            f"Adjusted cutoff: {original_cutoff} → {cutoff}"
        )

    train_cutoffs.append(cutoff)


assert len(set(train_cutoffs)) == N_TRAIN_WINDOWS
assert max(train_cutoffs) < validation_cutoff

assert test["date"].min() == test_cutoff + timedelta(days=1)
assert test["date"].max() == test_cutoff + timedelta(days=HORIZON)

print("Training cutoffs:", train_cutoffs[0], "to", train_cutoffs[-1])
print("Validation cutoff:", validation_cutoff)
print("Test cutoff:", test_cutoff)

Adjusted cutoff: 2016-12-18 → 2016-12-25
Training cutoffs: 2016-08-28 to 2017-07-02
Validation cutoff: 2017-07-30
Test cutoff: 2017-08-15


## 6. Build training, validation, and test feature tables

In [22]:
training_windows = []
for i, cutoff in enumerate(train_cutoffs, start=1):
    features = build_window(cutoff, train, train, include_target=True)
    training_windows.append(features)
    print(f'Built training window {i:02d}/{len(train_cutoffs)}: cutoff={cutoff}, rows={features.height:,}')

features_train = pl.concat(training_windows, how='vertical_relaxed')
features_validation = build_window(validation_cutoff, train, train, include_target=True)
features_test = build_window(test_cutoff, test, train, include_target=False)

print('features_train:     ', features_train.shape)
print('features_validation:', features_validation.shape)
print('features_test:      ', features_test.shape)

Built training window 01/12: cutoff=2016-08-28, rows=28,512
Built training window 02/12: cutoff=2016-09-25, rows=28,512
Built training window 03/12: cutoff=2016-10-23, rows=28,512
Built training window 04/12: cutoff=2016-11-20, rows=28,512
Built training window 05/12: cutoff=2016-12-25, rows=28,512
Built training window 06/12: cutoff=2017-01-15, rows=28,512
Built training window 07/12: cutoff=2017-02-12, rows=28,512
Built training window 08/12: cutoff=2017-03-12, rows=28,512
Built training window 09/12: cutoff=2017-04-09, rows=28,512
Built training window 10/12: cutoff=2017-05-07, rows=28,512
Built training window 11/12: cutoff=2017-06-04, rows=28,512
Built training window 12/12: cutoff=2017-07-02, rows=28,512
features_train:      (342144, 36)
features_validation: (28512, 36)
features_test:       (28512, 35)


## 7. Automated integrity checks

These assertions are intentionally strict. If one fails, do not continue to modeling until the cause is understood.

In [24]:
nonzero_nulls = (
    null_report
    .unpivot(
        index="dataset",
        variable_name="feature",
        value_name="null_count",
    )
    .filter(pl.col("null_count") > 0)
    .sort("dataset", "feature")
)

display(nonzero_nulls)

dataset,feature,null_count
str,str,u32
"""train""","""sales_lag_16""",1782
"""train""","""sales_lag_28""",1782


In [27]:
EXPECTED_HORIZON_ROWS = stores.height * train['family'].n_unique() * HORIZON
assert EXPECTED_HORIZON_ROWS == 28_512
assert features_validation.height == EXPECTED_HORIZON_ROWS
assert features_test.height == EXPECTED_HORIZON_ROWS
assert features_train.height == EXPECTED_HORIZON_ROWS * N_TRAIN_WINDOWS
assert 'sales' in features_train.columns and 'sales' in features_validation.columns
assert 'sales' not in features_test.columns

for frame_name, frame in [
    ('train', features_train),
    ('validation', features_validation),
    ('test', features_test),
]:
    duplicate_key = ['cutoff_date'] + KEYS if frame_name == 'train' else KEYS
    assert frame.select(pl.struct(duplicate_key).is_duplicated().any()).item() is False
    assert frame['day_of_forecast'].min() == 1
    assert frame['day_of_forecast'].max() == HORIZON
    assert frame.filter(pl.col('date') <= pl.col('cutoff_date')).is_empty()

critical_features = [
    'sales_mean_28', 'sales_mean_14', 'sales_mean_7', 'sales_std_28',
    'zero_rate_28', 'sales_lag_16', 'sales_lag_21', 'sales_lag_28',
    'onpromotion', 'type', 'cluster', 'day_of_week', 'day_of_forecast',
]
null_report = pl.concat([
    frame.select([pl.col(c).null_count().alias(c) for c in critical_features])
    .with_columns(pl.lit(name).alias('dataset'))
    for name, frame in [('train', features_train), ('validation', features_validation), ('test', features_test)]
], how='vertical_relaxed').select('dataset', *critical_features)

allowed_nullable_features = [
    "sales_lag_16",
    "sales_lag_21",
    "sales_lag_28",
    "sales_lag_35",
]

unexpected_nulls = (
    null_report
    .unpivot(
        index="dataset",
        variable_name="feature",
        value_name="null_count",
    )
    .filter(
        (pl.col("null_count") > 0)
        & ~pl.col("feature").is_in(allowed_nullable_features)
    )
)

display(unexpected_nulls)

assert unexpected_nulls.is_empty(), (
    f"Unexpected nulls found:\n{unexpected_nulls}"
)

print(
    "All integrity checks passed. "
    "Null values in lag features are allowed."
)

dataset,feature,null_count
str,str,u32


All integrity checks passed. Null values in lag features are allowed.


In [28]:
def add_lag_missing_indicators(frame):
    return frame.with_columns([
        pl.col(f"sales_lag_{lag}")
        .is_null()
        .cast(pl.Int8)
        .alias(f"sales_lag_{lag}_missing")
        for lag in SAFE_LAGS
    ])


features_train = add_lag_missing_indicators(features_train)
features_validation = add_lag_missing_indicators(features_validation)
features_test = add_lag_missing_indicators(features_test)

## 8. Inspect schema and sample

In [29]:
feature_columns = [c for c in features_train.columns if c not in ['sales', 'id']]
categorical_columns = ['family', 'city', 'state', 'type']

print(f'Feature columns ({len(feature_columns)}):')
print(feature_columns)
print('\nCategorical columns:', categorical_columns)
display(features_validation.head())
display(features_validation.select(
    'sales', 'sales_mean_28', 'sales_lag_16', 'onpromotion',
    'day_of_week', 'day_of_forecast', 'is_holiday'
).describe())

Feature columns (38):
['date', 'store_nbr', 'family', 'onpromotion', 'sales_mean_28', 'sales_std_28', 'sales_min_28', 'sales_max_28', 'zero_rate_28', 'sales_mean_14', 'sales_mean_7', 'sales_mean_prev7', 'sales_trend_7', 'sales_lag_16', 'sales_lag_21', 'sales_lag_28', 'sales_lag_35', 'city', 'state', 'type', 'cluster', 'dcoilwtico', 'is_holiday', 'has_holiday_type', 'has_event', 'holiday_count', 'cutoff_date', 'day_of_forecast', 'day_of_week', 'day_of_month', 'month', 'year', 'is_weekend', 'day_of_year', 'sales_lag_16_missing', 'sales_lag_21_missing', 'sales_lag_28_missing', 'sales_lag_35_missing']

Categorical columns: ['family', 'city', 'state', 'type']


id,date,store_nbr,family,sales,onpromotion,sales_mean_28,sales_std_28,sales_min_28,sales_max_28,zero_rate_28,sales_mean_14,sales_mean_7,sales_mean_prev7,sales_trend_7,sales_lag_16,sales_lag_21,sales_lag_28,sales_lag_35,city,state,type,cluster,dcoilwtico,is_holiday,has_holiday_type,has_event,holiday_count,cutoff_date,day_of_forecast,day_of_week,day_of_month,month,year,is_weekend,day_of_year,sales_lag_16_missing,sales_lag_21_missing,sales_lag_28_missing,sales_lag_35_missing
i64,date,i64,str,f64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,i64,f64,i8,i8,i8,i16,date,i8,i8,i8,i8,i16,i8,i16,i8,i8,i8,i8
2972376,2017-07-31,1,"""AUTOMOTIVE""",8.0,0,4.571429,2.847072,0.0,10.0,0.071429,4.785714,4.714286,4.857143,-0.142857,6.0,3.0,0.0,3.0,"""Quito""","""Pichincha""","""D""",13,50.21,0,0,0,0,2017-07-30,1,1,31,7,2017,0,212,0,0,0,0
2972377,2017-07-31,1,"""BABY CARE""",0.0,0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""Quito""","""Pichincha""","""D""",13,50.21,0,0,0,0,2017-07-30,1,1,31,7,2017,0,212,0,0,0,0
2972378,2017-07-31,1,"""BEAUTY""",3.0,0,3.25,1.77691,0.0,7.0,0.035714,2.785714,2.714286,2.857143,-0.142857,5.0,3.0,5.0,7.0,"""Quito""","""Pichincha""","""D""",13,50.21,0,0,0,0,2017-07-30,1,1,31,7,2017,0,212,0,0,0,0
2972379,2017-07-31,1,"""BEVERAGES""",2414.0,24,2179.428571,479.667588,1024.0,2859.0,0.0,2146.928571,2081.571429,2212.285714,-130.714286,2183.0,2338.0,2526.0,2242.0,"""Quito""","""Pichincha""","""D""",13,50.21,0,0,0,0,2017-07-30,1,1,31,7,2017,0,212,0,0,0,0
2972380,2017-07-31,1,"""BOOKS""",1.0,0,0.214286,0.498675,0.0,2.0,0.821429,0.071429,0.0,0.142857,-0.142857,0.0,0.0,1.0,0.0,"""Quito""","""Pichincha""","""D""",13,50.21,0,0,0,0,2017-07-30,1,1,31,7,2017,0,212,0,0,0,0


statistic,sales,sales_mean_28,sales_lag_16,onpromotion,day_of_week,day_of_forecast,is_holiday
str,f64,f64,f64,f64,f64,f64,f64
"""count""",28512.0,28512.0,28512.0,28512.0,28512.0,28512.0,28512.0
"""null_count""",0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",467.14295,473.418943,481.763042,5.924769,3.6875,8.5,0.064815
"""std""",1248.390215,1248.195868,1327.018836,15.905824,2.053044,4.609853,0.246203
"""min""",0.0,0.0,0.0,0.0,1.0,1.0,0.0
"""25%""",4.0,4.357143,3.0,0.0,2.0,5.0,0.0
"""50%""",30.0,30.857143,29.0,0.0,4.0,9.0,0.0
"""75%""",278.601,273.607143,279.009,6.0,5.0,12.0,0.0
"""max""",15190.0,11334.892857,18340.0,244.0,7.0,16.0,1.0


## 9. Save Parquet files

In [30]:
output_paths = {
    'train': INTERIM_DIR / 'features_train.parquet',
    'validation': INTERIM_DIR / 'features_validation.parquet',
    'test': INTERIM_DIR / 'features_test.parquet',
}

features_train.write_parquet(output_paths['train'], compression='zstd')
features_validation.write_parquet(output_paths['validation'], compression='zstd')
features_test.write_parquet(output_paths['test'], compression='zstd')

for name, path in output_paths.items():
    print(f'{name:10s} {path} ({path.stat().st_size / 1_048_576:.2f} MiB)')

train      retail-demand-forecasting/data/interim/features_train.parquet (5.11 MiB)
validation retail-demand-forecasting/data/interim/features_validation.parquet (0.45 MiB)
test       retail-demand-forecasting/data/interim/features_test.parquet (0.38 MiB)


## 10. What comes next

The next notebook, `04_lightgbm_model.ipynb`, will:

1. Load these three Parquet files.
2. Train one global LightGBM model using the historical windows.
3. Evaluate the untouched validation window with RMSLE.
4. Compare the result against the winning baseline: **mean_28 RMSLE = 0.521588**.
5. Report error by family and forecast day.
6. Train on all available labeled windows and create `submission.csv` only after validation succeeds.

A model is useful only if it beats the baseline on the same validation dates.